# SoccerNet no-export player+ball tracker

This notebook trains the lightweight heatmap detector/tracker directly from SoccerNet-Tracking's native MOT layout. It does **not** create a YOLO dataset, copy frames, or mirror `img1/` into another folder.

The model predicts two center heatmaps (`player`, `ball`) plus box width/height, then the tracker links detections frame-to-frame.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

DATA_DIR = '/content/drive/MyDrive/soccernet'
TRACKING_ROOT = f'{DATA_DIR}/tracking'
REPO_DIR = '/content/drive/MyDrive/repo'

# Keep training outputs on /content for speed, then copy checkpoints to Drive when done.
RUN_DIR = '/content/heatmap_tracker'
DRIVE_RUN_DIR = f'{DATA_DIR}/heatmap_tracker_runs'

!mkdir -p {DATA_DIR} {DRIVE_RUN_DIR}

## Install repo

This uses the current repo code on Drive. If the repo already exists, it pulls the latest changes.

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/atharvjain05/cv-project.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!pip install -q -r requirements-colab.txt
!pip install -q -r requirements-detect.txt
!pip install -q -e .

import sys
src = f'{REPO_DIR}/src'
if src not in sys.path:
    sys.path.insert(0, src)

## Check native SoccerNet tracking files

Expected layout:

```text
tracking/train/SNMOT-*/img1/000001.jpg
tracking/train/SNMOT-*/gt/gt.txt
tracking/train/SNMOT-*/gameinfo.ini
tracking/train/SNMOT-*/seqinfo.ini
```

In [ ]:
from pathlib import Path

root = Path(TRACKING_ROOT)
for split in ['train', 'test', 'challenge']:
    d = root / split
    if not d.is_dir():
        print(f'{split}: missing at {d}')
        continue
    seqs = sorted(p for p in d.iterdir() if p.is_dir() and (p / 'seqinfo.ini').is_file())
    print(f'{split}: {len(seqs)} sequences')
    if seqs:
        s = seqs[0]
        print(' example:', s.name)
        print('  image exists:', (s / 'img1' / '000001.jpg').is_file())
        print('  gt exists:   ', (s / 'gt' / 'gt.txt').is_file())
        print('  gameinfo:    ', (s / 'gameinfo.ini').is_file())

## Smoke-test the dataset

This loads a few frames directly from `img1/` and generates player+ball heatmap targets in memory from `gt.txt`.

In [ ]:
from soccernet_detect.direct_dataset import SoccerNetFrameDetections, CLASS_NAMES

ds = SoccerNetFrameDetections(
    TRACKING_ROOT,
    split='train',
    imgsz=512,
    output_stride=4,
    frame_stride=30,
    max_frames=16,
)
sample = ds[0]
print('classes:', CLASS_NAMES)
print('frames:', len(ds))
print('image:', tuple(sample['image'].shape))
print('heatmap:', tuple(sample['heatmap'].shape), 'max per class:', sample['heatmap'].flatten(1).max(dim=1).values.tolist())
print('box target cells:', int(sample['wh_mask'].sum().item()))

## Quick training run

This is meant to prove the pipeline works. It trains on a small subset with a high frame stride. Outputs stay on `/content` for faster writes.

In [ ]:
# %cd {REPO_DIR}
# !python -m soccernet_detect.train_heatmap_tracker \
#   --tracking-root {TRACKING_ROOT} \
#   --split train \
#   --out {RUN_DIR} \
#   --epochs 5 \
#   --imgsz 512 \
#   --batch 8 \
#   --frame-stride 30 \
#   --max-frames 2000 \
#   --device cuda

## Fuller training run

After the quick run works, lower `--frame-stride` and remove or raise `--max-frames`. This still does not export or copy the dataset.

In [ ]:
# Example fuller run. Uncomment when ready.
%cd {REPO_DIR}
!python -m soccernet_detect.train_heatmap_tracker \
  --tracking-root {TRACKING_ROOT} \
  --split train \
  --val-split test \
  --out {RUN_DIR}_full \
  --epochs 25 \
  --imgsz 512 \
  --batch 256 \
  --frame-stride 5 \
  --device cuda \
  --max-frames 10000 \
  --max-val-frames 2000 \
  --num-workers 8

## Save checkpoint to Drive

Training writes to `/content` for speed. Copy the checkpoint/history to Drive when the run is done.

In [ ]:
!mkdir -p {DRIVE_RUN_DIR}
!cp -r {RUN_DIR} {DRIVE_RUN_DIR}/heatmap_tracker_latest
!ls -lh {DRIVE_RUN_DIR}/heatmap_tracker_latest

## What the model is learning

For each MOT box, the dataset computes the object center and draws a small Gaussian on one of two heatmap channels: `player` or `ball`. The model also regresses the box width/height at the true center cells.

Loss:

```text
loss = focal_heatmap_loss(player+ball centers) + box_weight * L1(width,height at true centers)
```

At inference, heatmap peaks become detections. The simple tracker links player boxes with IoU/center distance, and links the ball with nearest-center distance because ball boxes are tiny.